# 01 — Data Collection & Curation

**Project:** *Paid for What? Measuring how NBA salaries track on-court performance* — DS 325, Fall 2026

**Research question.** Which measurable aspects of performance does the NBA labor market actually pay for,
and which players are over- or under-paid relative to their production?

**Plan.** (1) Build a player-season dataset joining performance, salary, roster, and draft data; (2) model a player's
salary (as a share of the salary cap) from information available *before* the season; (3) use the gap between actual and
predicted salary as a **surplus-value** measure, and test whether teams with more surplus value win more.

This notebook documents where the data comes from and how it was collected. The collection code lives in `src/`:

| Step | Script | What it does |
|---|---|---|
| 1 | `src/download.py` | Downloads 389 pages from Basketball-Reference into a local gzip cache (rate-limited to stay under the site's 20 req/min rule; resumable) |
| 2 | `src/parse_bref.py` | Parses the HTML tables into tidy CSVs in `data/raw/` |
| 3 | `src/build_dataset.py` | Joins everything into one row per player-season → `data/processed/player_seasons.csv` |

Run all three in order to reproduce the dataset from scratch (~45 minutes, dominated by the polite crawl delay).

## Data source

All data comes from **Basketball-Reference.com** (Sports Reference LLC), the standard public reference for NBA
statistics. Sports Reference compiles box-score data from official NBA game records and publishes salary data
compiled from public contract reporting. Citation:

> Sports Reference LLC. *Basketball-Reference.com — Basketball Statistics and History.* https://www.basketball-reference.com/ (accessed November 2026).

| Table | Pages | Source URL pattern | Grain |
|---|---|---|---|
| Per-game stats | 11 | `/leagues/NBA_{season}_per_game.html` | player × season × team stint |
| Advanced stats | 11 | `/leagues/NBA_{season}_advanced.html` | player × season × team stint |
| Salaries | 330 | `/teams/{team}/{season}.html` (salary table) | player × season × team |
| Rosters (height, weight, experience) | (same pages) | `/teams/{team}/{season}.html` (roster table) | player × season × team |
| Team records | (same pages) | `/teams/{team}/{season}.html` | team × season |
| Draft history | 36 | `/draft/NBA_{year}.html`, 1990–2025 | player |
| Salary cap history | 1 | `/contracts/salary-cap-history.html` | season |

**Seasons covered:** 2015-16 through 2025-26 (11 seasons; Basketball-Reference labels seasons by their end year,
so "2016" means 2015-16). All 30 franchises.

**Why scrape instead of an API?** Free NBA APIs (e.g. `balldontlie`, `nba_api`) cover stats but **not salaries**,
and the salary sites we tried (ESPN, HoopsHype) block scripted access. Basketball-Reference has stats, salaries,
rosters and draft data under one consistent **player id** (e.g. `curryst01`), so every join is exact rather than
fuzzy name matching.

In [1]:
import sys
sys.path.append("../src")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from features import build_features
import plotting as P

P.setup()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if abs(x) >= 1000 else f"{x:,.3f}")

In [2]:
raw = {name: pd.read_csv(f"../data/raw/{name}.csv")
       for name in ["per_game", "advanced", "salaries", "rosters", "teams", "draft", "salary_cap"]}
pd.DataFrame({name: {"rows": len(d), "columns": d.shape[1]} for name, d in raw.items()}).T

,rows,columns
per_game,7595,33
advanced,7595,31
salaries,7299,5
rosters,6819,11
teams,330,4
draft,2101,6
salary_cap,43,3


In [3]:
raw["salaries"].sample(5, random_state=1).style.format({"salary": "${:,.0f}"}).hide(axis="index")

season,team,player_id,player,salary
2018,BRK,crabbal01,Allen Crabbe,"$19,332,500"
2024,WAS,coulibi01,Bilal Coulibaly,"$6,614,280"
2018,BRK,allenja01,Jarrett Allen,"$1,713,720"
2025,ORL,wagnefr01,Franz Wagner,"$7,007,092"
2022,DET,grantje01,Jerami Grant,"$20,002,500"


In [4]:
raw["per_game"][["season", "player_id", "name_display", "team_name_abbr", "games", "mp_per_g", "pts_per_g"]].head().style.hide(axis="index")

season,player_id,name_display,team_name_abbr,games,mp_per_g,pts_per_g
2016,curryst01,Stephen Curry,GSW,79,34.200000,30.100000
2016,hardeja01,James Harden,HOU,82,38.100000,29.000000
2016,duranke01,Kevin Durant,OKC,72,35.800000,28.200000
2016,couside01,DeMarcus Cousins,SAC,65,34.600000,26.900000
2016,jamesle01,LeBron James,CLE,76,35.600000,25.300000


In [5]:
raw["salary_cap"].query("season >= 2016").style.format({"salary_cap": "${:,.0f}"}).hide(axis="index")

season,year_id,salary_cap
2016,2015-16,"$70,000,000"
2017,2016-17,"$94,143,000"
2018,2017-18,"$99,093,000"
2019,2018-19,"$101,869,000"
2020,2019-20,"$109,140,000"
2021,2020-21,"$109,140,000"
2022,2021-22,"$112,414,000"
2023,2022-23,"$123,655,000"
2024,2023-24,"$136,021,000"
2025,2024-25,"$140,588,000"


## Curation decisions (implemented in `src/build_dataset.py`)

1. **Traded players.** A player traded mid-season has one stat line per team plus a combined line (`2TM`, `3TM`).
   We keep the combined line, record the number of teams, and assign the player to the **last** team he played for.
2. **Salary for traded / waived players.** Team salary tables also list *dead money* — salary still owed to players a
   team waived. So a player's salary is the **max listed by a team he actually played for** that season; only if no
   such listing exists do we fall back to any listing (`salary_source` records which).
3. **Exact joins on player id.** Stats ↔ salary ↔ roster ↔ draft are joined on Basketball-Reference player ids.
4. **&#36;0 salaries.** Some two-way-contract players are listed at &#36;0 because Basketball-Reference has no amount for
   them; those listings are treated as missing salary rather than as a real &#36;0.
5. **Undrafted players.** Players with no draft record (1990–2025) are flagged `undrafted`.
6. **Salary normalization.** The salary cap more than doubled from &#36;70.0M (2015-16) to &#36;154.6M (2025-26), so the
   modeling target is **salary as a share of that season's cap** (`salary_pct_cap`), not raw dollars.

In [6]:
ps = pd.read_csv("../data/processed/player_seasons.csv")
print(f"{len(ps):,} player-seasons | {ps.player_id.nunique():,} unique players | "
      f"{ps.has_salary.sum():,} with a salary record ({ps.has_salary.mean():.1%})")
assert not ps.duplicated(["season", "player_id"]).any(), "duplicate player-season rows"
(ps.groupby("season").agg(players=("player_id", "size"), with_salary=("has_salary", "sum"),
                          median_salary=("salary", "median"), salary_cap=("salary_cap", "first"))
   .style.format({"median_salary": "${:,.0f}", "salary_cap": "${:,.0f}"}))

5,968 player-seasons | 1,554 unique players | 5,419 with a salary record (90.8%)


,players,with_salary,median_salary,salary_cap
season,,,,
2016,476,476,"$2,568,830","$70,000,000"
2017,486,484,"$3,162,022","$94,143,000"
2018,540,483,"$3,097,800","$99,093,000"
2019,530,482,"$3,260,917","$101,869,000"
2020,529,480,"$3,349,300","$109,140,000"
2021,540,481,"$3,640,200","$109,140,000"
2022,605,562,"$2,641,691","$112,414,000"
2023,539,485,"$4,000,000","$123,655,000"
2024,572,499,"$4,310,250","$136,021,000"


**Sample size.** The processed table has 5,968 player-seasons — well over the 2,000-sample threshold, so no
small-sample justification is needed. Restricting to players with a salary and at least 250 minutes (see notebook 03)
still leaves 4,488 player-seasons, and the market-value model sample (non-rookie contracts with a previous season)
is about 2,800.